In [1]:
import pandas as pd
from datetime import timedelta,datetime

In [2]:
# 设置列表最大行数为500 
pd.set_option('display.max_rows', 500)  
pd.set_option('display.width', 1000)  
pd.set_option('display.max_colwidth', 1000) 

In [ ]:
# 将tab替换为空格，将替换连续的两个空格为一个空格,直到没有连续的空格以便于后面代码读取txt并以空格分列内容
def replace_double_spaces(file_path):
    with open(file_path, 'r') as file:
        content = file.read()
    previous_length = len(content)
    while True:
        # tab替换为空格，替换连续的两个空格为一个空格
        content = content.replace("	", " ")
        content = content.replace("  ", " ")
        current_length = len(content)
        if current_length == previous_length:
            break
        previous_length = current_length    
    with open(file_path, 'w') as file:
        file.write(content)

In [3]:
#读取数据集名，中间过程，输出数据集名，表格列名['datatime_ymd','datatime_hms','did','d_value','activity','1','2','3','4','5','6','7']
filename = 'hh130.rawdata.txt'
filename_first = 'hh130.csv'
headers = ['Date','Time','Sensor','Translate01','Translate02','Message','SensorType']

In [ ]:
#调替换函数
replace_double_spaces(filename)

In [ ]:
#打开数据集
dataset = pd.read_csv(filename,delimiter=' ', header=None,names = headers)
#报错ParserError: Error tokenizing data. C error: Expected 6 fields in line 69, saw 8，取决于第一行定义了几列
dataset.head()

In [ ]:
# #判断activity要加几行
# activity1234 = data[data['2'].apply(lambda x: type(x) != float)]
# activity1234

In [ ]:
#提取年月日和hms到列表
date = dataset['Date'].tolist() 
time = dataset['Time'].tolist() 

In [ ]:
#以\t合并年月日和hms，并放入表格中修整格式
date_time = []
for i in range(0,len(date)) :
    datetime_list = [date[i],time[i]]
    datetime_add = ' '.join(datetime_list)
    date_time.append(datetime_add)
dataset.rename(columns={'Date': 'Datetime'}, inplace=True)#重命名列名
dataset.loc[:,'Datetime'] = date_time#将修约好的时间放回df
dataset.drop(['Time'],axis = 1,inplace = True)#删除不需要的列
dataset.head()

In [ ]:
# 将不带毫秒的e_datetime增加毫秒
dataset.loc[dataset['Datetime'].str.contains(r'^[^.]*$'), 'Datetime']+= '.0'

In [ ]:
#对时间格式处理的函数
def round_timestamp(s):
    s = s.apply(lambda x: datetime.strptime(x, '%Y-%m-%d %H:%M:%S.%f'))
    seconds = s.dt.microsecond  # 获取小数部分（秒）
    max_digits = seconds.map(lambda x: len(str(x))).max()
    seconds0 = seconds.map(lambda x: format(x / (10 ** max_digits), f'.{max_digits}f'))
    rounded_seconds = round(seconds0.astype(float))  # 四舍五入修约
    s = s.apply(lambda x: x.replace(microsecond=0))  # 将微秒部分置零
    s += pd.to_timedelta(rounded_seconds, unit='s')  # 加上修约后的秒数
    # 处理分界线
    #s = s.apply(lambda x: x.replace(year=2077))  # 将年份替换为2077
    #s = s.apply(lambda x: x.timestamp())  # 转换为timestamp数字
    return s

In [ ]:
time = round_timestamp(dataset['Datetime'].astype(str))

In [ ]:
dataset.loc[:,'Datetime'] = time#将修约好的时间放回df
dataset.head()

In [ ]:
#保存为csv格式
dataset.to_csv(filename_first,index = False)

In [4]:
# 读取CSV文件
dataset= pd.read_csv(filename_first)
dataset.head()

,Datetime,Sensor,Translate01,Translate02,Message,SensorType
0,2014-04-17 00:02:31,ZB106,Ignore,Ignore,OK,Control4-Radio
1,2014-04-17 00:04:24,ZB102,Ignore,Ignore,OK,Control4-Radio
2,2014-04-17 00:05:16,ZB010,Ignore,Ignore,OK,Control4-Radio
3,2014-04-17 00:05:19,ZB009,Ignore,Ignore,OK,Control4-Radio
4,2014-04-17 00:07:08,ZB103,Ignore,Ignore,OK,Control4-Radio


In [5]:
#读取Sensor,Translate01列的所有类型,去重并排列
sensor = dataset.loc[:, ['Sensor']] #读取
translate01 = dataset.loc[:, [ 'Translate01']]
sensor_id = sensor.drop_duplicates()#去重
translate01_id = translate01.drop_duplicates()
sensor_id =sensor_id.sort_values(by = 'Sensor',ascending = True)#以sensor这列排序
translate01_id= translate01_id.sort_values(by = 'Translate01',ascending = True)
sensor_id = sensor_id.reset_index(drop = True)#索引重排
translate01_id = translate01_id.reset_index(drop = True)
print(sensor_id)
print(translate01_id)

     Sensor
0   BATP001
1   BATP002
2   BATP003
3   BATP004
4   BATP005
5   BATP006
6   BATP007
7   BATP008
8   BATP009
9   BATP010
10  BATP011
11  BATP102
12  BATP103
13  BATP104
14     D002
15    LS001
16    LS002
17    LS003
18    LS004
19    LS005
20    LS006
21    LS007
22    LS008
23    LS009
24    LS010
25    LS011
26     M001
27     M002
28     M003
29     M004
30     M005
31     M006
32     M011
33    MA007
34    MA008
35    MA009
36    MA010
37     T102
38     T103
39     T104
40     T106
41    ZB001
42    ZB002
43    ZB003
44    ZB004
45    ZB005
46    ZB006
47    ZB007
48    ZB008
49    ZB009
50    ZB010
51    ZB011
52    ZB102
53    ZB103
54    ZB104
55    ZB106
   Translate01
0     Bathroom
1      Bedroom
2       Ignore
3      Kitchen
4   LivingRoom
5  OutsideDoor


In [6]:
#检查Sensor,Translate01列的特殊类型，并判断是否准备洗去,此处为可视化处理无实际意义
button = dataset[dataset['Sensor'].str.contains('Button')]#模糊检索带button
button = button.loc[:, ['Sensor', 'Translate01', 'Translate02', 'Message', 'SensorType']]#去掉时间
button = button.drop_duplicates()#去重
#button = sensor_type.sort_values(by = 'Sensor',ascending = True)#以sensor这列排序
print(button)

Empty DataFrame
Columns: [Sensor, Translate01, Translate02, Message, SensorType]
Index: []


In [7]:
#模糊检索要洗去的类型
wash_env = sensor_id[sensor_id['Sensor'].str.contains('BATP')| \
                      sensor_id['Sensor'].str.contains('ZB')| \
                      sensor_id['Sensor'].str.contains('ZBL')]
wash_env = wash_env.reset_index(drop = True)#索引重排
#保存为csv格式
wash_env.to_csv('wash_env.csv',index = False)
#wash_env

In [8]:
#删除洗去的类型
env = sensor_id[~sensor_id['Sensor'].isin(wash_env['Sensor'])]
env = env.reset_index(drop = True)#索引重排
env.columns = ['D_ID']#修改列名
env.to_csv('env.csv',index = False)

In [9]:
def data_trans(data, trans_relationship):
    for key in trans_relationship.keys():
        for column_key in trans_relationship[key].keys():
            data.loc[data['d_name'].str[0:2] == key, column_key] = trans_relationship[key][column_key]
    return data[['level','d_type','d_func','d_name']]

def room_trans(data, trans_relationship):
    for key in trans_relationship.keys():
        data[key] = trans_relationship[key]
    return data[['level','d_type','d_func','d_name']]

def data_comb(data1, data2):
    data = pd.concat([data1, data2]).sort_values(by=['level', 'd_type'])
    return data.reset_index(drop=True)

In [10]:
data = pd.read_csv('env.csv')
data = data.rename(columns = {'D_ID': 'd_name'})

In [11]:
rooms = translate01_id['Translate01'].tolist()#提取房间名为列表
df_rooms = pd.DataFrame(rooms, columns=['d_func'])

In [12]:
#把各对应关系编入字典
trans_relationship = {
    'D0': {'level': 2,   'd_type': 'sensor',     'd_func': 'door'       },
    'LS': {'level': 2,   'd_type': 'sensor',     'd_func': 'light'      },
    'M0': {'level': 2,   'd_type': 'sensor',     'd_func': 'motion'     },
    'MA': {'level': 2,   'd_type': 'sensor',     'd_func': 'ambient'    },
    'T1': {'level': 2,   'd_type': 'sensor',     'd_func': 'temperature'},
    'L0': {'level': 2,   'd_type': 'sensor',     'd_func': 'lamp'       },
    'Bu': {'level': 3,   'd_type': 'controller', 'd_func': 'button'     }
}
rooms_trans_relationship = {'level': 1,   'd_type': 'room',     'd_name': '/'       }

In [13]:
data1 = room_trans(df_rooms, rooms_trans_relationship)
data2 = data_trans(data, trans_relationship)

In [14]:
env = data_comb(data1, data2)
env.to_csv('env.csv',index = False)

In [15]:
# 读取CSV文件
data_env = pd.read_csv('env.csv')
data_env.head(10)

,level,d_type,d_func,d_name
0,1.0,room,Bathroom,/
1,1.0,room,Bedroom,/
2,1.0,room,Ignore,/
3,1.0,room,Kitchen,/
4,1.0,room,LivingRoom,/
5,1.0,room,OutsideDoor,/
6,2.0,sensor,door,D002
7,2.0,sensor,light,LS001
8,2.0,sensor,light,LS002
9,2.0,sensor,light,LS003


In [16]:
#重命名数据集的Datetime，Sensor，Message列，找到数据集中d_name列冗余数据
dataset = dataset.rename(columns = {'Datetime': 'ts','Sensor': 'd_name','Message': 'd_value'})
d_name = data_env['d_name'].tolist()
error_d_name = dataset[~dataset['d_name'].isin(d_name)]
error_d_name         

,ts,d_name,Translate01,Translate02,d_value,SensorType
0,2014-04-17 00:02:31,ZB106,Ignore,Ignore,OK,Control4-Radio
1,2014-04-17 00:04:24,ZB102,Ignore,Ignore,OK,Control4-Radio
2,2014-04-17 00:05:16,ZB010,Ignore,Ignore,OK,Control4-Radio
3,2014-04-17 00:05:19,ZB009,Ignore,Ignore,OK,Control4-Radio
4,2014-04-17 00:07:08,ZB103,Ignore,Ignore,OK,Control4-Radio
...,...,...,...,...,...,...
1156628,2014-12-23 15:35:51,ZB010,Ignore,Ignore,OK,Control4-Radio
1156650,2014-12-23 15:36:43,ZB106,Ignore,Ignore,OK,Control4-Radio
1156651,2014-12-23 15:36:45,ZB004,Ignore,Ignore,OK,Control4-Radio
1156746,2014-12-23 15:38:35,ZB003,Ignore,Ignore,OK,Control4-Radio


In [17]:
#模糊检索要洗去Button的TAP_COUNT，RELEASE
drop_button = dataset[dataset['d_value'].str.contains('TAP_COUNT')| \
                      dataset['d_value'].str.contains('HOLD_DEPRESS')| \
                      dataset['d_value'].str.contains('HOLD_RELEASE')| \
                      dataset['d_value'].str.contains('RELEASE')]
drop_button.head(10)

,ts,d_name,Translate01,Translate02,d_value,SensorType


In [19]:
#合并需洗去的数据  
wash_data  = pd.concat([error_d_name ,drop_button])

In [22]:
# 将d_value的单一判断类结果值转换为0/1数值
dataset['d_value'] = dataset['d_value'].replace({"ON": 1.0, "OFF": 0.0, "OPEN": 1.0, "CLOSE": 0.0})
dataset.head(20)

,ts,d_name,Translate01,Translate02,d_value,SensorType
0,2014-04-17 00:27:53,T104,Ignore,KitchenTemp,19,Control4-Temperature
1,2014-04-17 06:39:28,LS003,Ignore,Ignore,03,Control4-LightSensor
2,2014-04-17 06:39:31,LS006,Ignore,Ignore,01,Control4-LightSensor
3,2014-04-17 06:39:41,LS005,Ignore,Ignore,01,Control4-LightSensor
4,2014-04-17 06:39:42,LS001,Ignore,Ignore,01,Control4-LightSensor
5,2014-04-17 06:42:33,LS008,Ignore,Ignore,02,Control4-LightSensor
6,2014-04-17 06:42:40,LS007,Ignore,Ignore,01,Control4-LightSensor
7,2014-04-17 06:47:28,LS009,Ignore,Ignore,01,Control4-LightSensor
8,2014-04-17 06:49:28,LS003,Ignore,Ignore,05,Control4-LightSensor
9,2014-04-17 06:49:30,LS006,Ignore,Ignore,03,Control4-LightSensor


In [23]:
# 搜索d_value的未转换成01的数据
character_values = dataset[dataset['d_value'].apply(lambda x: isinstance(x, str))]['d_value']
character_values

0         19
1         03
2         01
3         01
4         01
          ..
957330    10
957333    09
957335    10
957373    09
957379    08
Name: d_value, Length: 206767, dtype: object

In [24]:
# 去除重复值，得到唯一的字符数据列表
my_list = list(character_values.unique())
f_list = my_list.copy()
print(f_list)
for value in my_list:
    try:
        if float(value)<=60:
            f_list.pop(f_list.index(value))
    except:
        pass

error_value = pd.DataFrame()
for charv in f_list:
    error_value = pd.concat([error_value, dataset[dataset['d_value'] == charv]])
error_value

['19', '03', '01', '02', '05', '04', '06', '09', '07', '22', '00', '08', '20', '10', '41', '33', '34', '12', '35', '11', '42', '14', '13', '24', '43', '36', '23', '25', '26', '15', '16', '17', '18', '52', '51', '32', '31', '30', '29', '50', '21', '28', '53', '39', '40', '27', '45', '46', '47', '44', '37', '49', '48', '38', '54', '56', '55', '57', '58', '60', '59', '61', '62', '64', '63', '65', '68', '66', '67', '69', '98', '24.0', '23.0', '23.5', '24.5', '25.0', '25.5', '22.5', '22.0', '21.5', '21.0', '20.5', '20.0', '19.5', '26.0', '26.5']


,ts,d_name,Translate01,Translate02,d_value,SensorType
214451,2014-07-27 09:13:10,LS006,Ignore,Ignore,61,Control4-LightSensor
246130,2014-08-02 10:39:14,LS006,Ignore,Ignore,61,Control4-LightSensor
295785,2014-08-12 10:13:25,LS006,Ignore,Ignore,61,Control4-LightSensor
366737,2014-08-25 10:08:09,LS006,Ignore,Ignore,61,Control4-LightSensor
366831,2014-08-25 10:48:09,LS006,Ignore,Ignore,61,Control4-LightSensor
404858,2014-09-01 11:05:15,LS006,Ignore,Ignore,61,Control4-LightSensor
404859,2014-09-01 11:05:16,LS007,Ignore,Ignore,61,Control4-LightSensor
404984,2014-09-01 12:42:27,LS006,Ignore,Ignore,61,Control4-LightSensor
405057,2014-09-01 12:47:28,LS006,Ignore,Ignore,61,Control4-LightSensor
405240,2014-09-01 12:57:18,LS006,Ignore,Ignore,61,Control4-LightSensor


In [ ]:
#考虑要不要再次合并需洗去的数据  
#wash_data  = pd.concat([wash_data ,error_value])

In [20]:
# 清洗d_name异常数据
dataset = dataset.drop(wash_data.index)

In [21]:
#数据集索引重排
dataset = dataset.reset_index(drop = True)
dataset

,ts,d_name,Translate01,Translate02,d_value,SensorType
0,2014-04-17 00:27:53,T104,Ignore,KitchenTemp,19,Control4-Temperature
1,2014-04-17 06:39:28,LS003,Ignore,Ignore,03,Control4-LightSensor
2,2014-04-17 06:39:31,LS006,Ignore,Ignore,01,Control4-LightSensor
3,2014-04-17 06:39:41,LS005,Ignore,Ignore,01,Control4-LightSensor
4,2014-04-17 06:39:42,LS001,Ignore,Ignore,01,Control4-LightSensor
...,...,...,...,...,...,...
957394,2014-12-23 15:41:35,M003,LivingRoom,LivingRoom,OFF,Control4-Motion
957395,2014-12-23 15:41:37,M003,LivingRoom,LivingRoom,ON,Control4-Motion
957396,2014-12-23 15:41:38,M003,LivingRoom,LivingRoom,OFF,Control4-Motion
957397,2014-12-23 15:41:48,M003,LivingRoom,LivingRoom,ON,Control4-Motion


In [25]:
LEVEL = data_env['level'].tolist() #提取该列内容生成列表
D_TYPE = data_env['d_type'].tolist()
D_FUNC = data_env['d_func'].tolist() 
D_NAME = data_env['d_name'].tolist()
#COORDINATE = data_information['COORDINATE'].tolist()
#LAYER = data_information['LAYER'].tolist()
d_name = dataset['d_name'].tolist()

In [26]:
# 建立信息规则字典,LAYER[level]
information = {}
for level in range(0,len(D_NAME)) :
    information[D_NAME[level]]= [D_FUNC[level],LEVEL[level],D_TYPE[level]]  

In [27]:
# 匹配信息并存入数据集
add_information = []
for i_d_name in d_name :
    add_information.append(information[i_d_name])
add_information

[['temperature', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['temperature', 2.0, 'sensor'],
 ['motion', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['temperature', 2.0, 'sensor'],
 ['motion', 2.0, 'sensor'],
 ['light', 2.0, 'sensor'],
 ['ambient', 2.0, 'sensor'],
 ['lig

In [28]:
#生产data_add的df
dataset_add = pd.DataFrame(data = add_information,columns =['d_func','level','d_type'])
dataset_add

,d_func,level,d_type
0,temperature,2.0,sensor
1,light,2.0,sensor
2,light,2.0,sensor
3,light,2.0,sensor
4,light,2.0,sensor
...,...,...,...
957394,motion,2.0,sensor
957395,motion,2.0,sensor
957396,motion,2.0,sensor
957397,motion,2.0,sensor


In [29]:
#匹配并加入'd_func','level','d_type'列
dataset_final = pd.merge(dataset,dataset_add,right_index =True,left_index = True)
dataset_final

,ts,d_name,Translate01,Translate02,d_value,SensorType,d_func,level,d_type
0,2014-04-17 00:27:53,T104,Ignore,KitchenTemp,19,Control4-Temperature,temperature,2.0,sensor
1,2014-04-17 06:39:28,LS003,Ignore,Ignore,03,Control4-LightSensor,light,2.0,sensor
2,2014-04-17 06:39:31,LS006,Ignore,Ignore,01,Control4-LightSensor,light,2.0,sensor
3,2014-04-17 06:39:41,LS005,Ignore,Ignore,01,Control4-LightSensor,light,2.0,sensor
4,2014-04-17 06:39:42,LS001,Ignore,Ignore,01,Control4-LightSensor,light,2.0,sensor
...,...,...,...,...,...,...,...,...,...
957394,2014-12-23 15:41:35,M003,LivingRoom,LivingRoom,0.0,Control4-Motion,motion,2.0,sensor
957395,2014-12-23 15:41:37,M003,LivingRoom,LivingRoom,1.0,Control4-Motion,motion,2.0,sensor
957396,2014-12-23 15:41:38,M003,LivingRoom,LivingRoom,0.0,Control4-Motion,motion,2.0,sensor
957397,2014-12-23 15:41:48,M003,LivingRoom,LivingRoom,1.0,Control4-Motion,motion,2.0,sensor


In [30]:
#生成由'ts','d_name','d_type','d_func','d_value'列组成的data
dataset_data = dataset_final[['ts','d_name','d_type','d_func','d_value']]

In [31]:
#模糊检索要env需要合并的ButtonUp和ButtonDown并删除
drop_data_env = data_env[data_env['d_name'].str.contains('ButtonDown')]
data_env = data_env.drop(drop_data_env.index)

In [32]:
#重命名ButtonUp为Button
data_env['d_name'] = data_env['d_name'].str.replace('ButtonUp', 'Button') 
data_env.to_csv('env.csv',index = False)

In [33]:
#将'd_name'列带有字符串'Buttonup'的行所对应的'd_value'列下该行的数据改为浮点数1。down为0
dataset_data.loc[dataset_data['d_name'].str.contains('ButtonUp'), 'd_value'] = 1.0
dataset_data.loc[dataset_data['d_name'].str.contains('ButtonDown'), 'd_value'] = 0

C:\Users\78785\AppData\Local\Temp\ipykernel_16884\2643777897.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset_data.loc[dataset_data['d_name'].str.contains('ButtonUp'), 'd_value'] = 1.0
C:\Users\78785\AppData\Local\Temp\ipykernel_16884\2643777897.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset_data.loc[dataset_data['d_name'].str.contains('ButtonDown'), 'd_value'] = 0


In [34]:
#重命名Button
dataset_data['d_name'] = dataset_data['d_name'].str.replace('ButtonUp', 'Button')
dataset_data['d_name'] = dataset_data['d_name'].str.replace('ButtonDown', 'Button')

C:\Users\78785\AppData\Local\Temp\ipykernel_16884\3038134514.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset_data['d_name'] = dataset_data['d_name'].str.replace('ButtonUp', 'Button')
C:\Users\78785\AppData\Local\Temp\ipykernel_16884\3038134514.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset_data['d_name'] = dataset_data['d_name'].str.replace('ButtonDown', 'Button')


In [35]:
#最终保存为csv格式
dataset_data.to_csv('data.csv',index = False)
wash_data.to_csv('wash.csv',index = False)